In [5]:
# ===============================================================
# YikYak Sentiment Fine-Tuning — Optimized Edition
# Base: cardiffnlp/twitter-roberta-base-sentiment-latest
# Labels: 0 = Negative | 1 = Neutral | 2 = Positive
# ===============================================================

import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import spacy
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────
BASE_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
SAVE_PATH  = "yikyak_sentiment_model_v2"
DATA_PATH  = "yikyak_labeled_final.csv"
TEXT_COL   = "text"
LABEL_COL  = "label"

# ── Hyperparameters (optimized) ────────────────────────────────
EPOCHS        = 3
BATCH_SIZE    = 128
LR            = 2e-5
MAX_LEN       = 64
GRAD_CLIP     = 1.0
WARMUP_PCT    = 0.1
WEIGHT_DECAY  = 0.01
TEST_SIZE     = 0.2
SEED          = 42
NUM_WORKERS   = os.cpu_count()

LABEL_NAMES = ["Negative", "Neutral", "Positive"]

# ── Device selection (CUDA / MPS / CPU) ────────────────────────
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

USE_AMP = device.type == "cuda"

print(f"Device: {device}")

# ── Reproducibility ────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ── Load dataset ───────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=[TEXT_COL, LABEL_COL])

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(int)

print(f"\nTotal posts: {len(df)}")

for lbl, cnt in sorted(Counter(df[LABEL_COL]).items()):
    print(f"{LABEL_NAMES[lbl]} ({lbl}) : {cnt}")

counts = np.array([Counter(df[LABEL_COL])[i] for i in range(3)], dtype=float)
class_weights = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float).to(device)

# ── Train/Test split ───────────────────────────────────────────
sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(sss.split(df[TEXT_COL], df[LABEL_COL]))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

print(f"\nTrain: {len(train_df)} | Test: {len(test_df)}")

# ── Tokenizer ──────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_function(examples):

    return tokenizer(
        examples[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

# ── Convert to HF Dataset ──────────────────────────────────────
train_ds = Dataset.from_pandas(train_df[[TEXT_COL, LABEL_COL]])
test_ds  = Dataset.from_pandas(test_df[[TEXT_COL, LABEL_COL]])

train_ds = train_ds.map(tokenize_function, batched=True, num_proc=NUM_WORKERS)
test_ds  = test_ds.map(tokenize_function, batched=True, num_proc=NUM_WORKERS)

train_ds = train_ds.rename_column(LABEL_COL, "labels")
test_ds  = test_ds.rename_column(LABEL_COL, "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# ── DataLoaders ────────────────────────────────────────────────
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ── Model ──────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True
)

model.to(device)

print("torch.compile disabled — standard PyTorch")

# ── Optimizer & Scheduler ──────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS

scheduler = OneCycleLR(
    optimizer,
    max_lr=LR,
    total_steps=total_steps,
    pct_start=WARMUP_PCT,
    anneal_strategy="cos"
)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

scaler = GradScaler(enabled=USE_AMP)

# ── Evaluation ─────────────────────────────────────────────────
def evaluate(loader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with autocast(enabled=USE_AMP):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return acc, f1

# ── Training Loop ──────────────────────────────────────────────
print("\nStarting Training\n")

best_f1 = 0

for epoch in range(1, EPOCHS + 1):

    model.train()
    total_loss = 0

    bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", mininterval=2)

    for batch in bar:

        optimizer.zero_grad(set_to_none=True)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with autocast(enabled=USE_AMP):

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_fn(outputs.logits, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        total_loss += loss.item()

        bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    val_acc, val_f1 = evaluate(test_loader)

    print(f"\nEpoch {epoch}")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Val Acc: {val_acc:.4f}")
    print(f"Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:

        best_f1 = val_f1

        model.save_pretrained(SAVE_PATH)
        tokenizer.save_pretrained(SAVE_PATH)

        print(f"Model saved → {SAVE_PATH}")

Device: mps

Total posts: 32768
Negative (0) : 8997
Neutral (1) : 19904
Positive (2) : 3867

Train: 26214 | Test: 6554


Map (num_proc=8): 100%|██████████| 6554/6554 [00:01<00:00, 5513.66 examples/s]
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


torch.compile disabled — standard PyTorch

Starting Training



Epoch 1/3:   0%|          | 0/205 [00:00<?, ?it/s]Process Process-31:
Process Process-34:
Process Process-30:
Process Process-32:
Process Process-38:
Process Process-33:
Process Process-37:
Process Process-27:
Process Process-35:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/multiprocessing/process.py", line 318, in _bootstrap
    util._exit_function()
KeyboardInterrupt
Process Process-26:
Process Process-36:
Process Process-39:
Process Process-25:
Process Process-28:
Process Process-40:
Process Process-29:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
KeyboardInterrupt
Traceback (most recent call last):
Traceback (most recent call last):

KeyboardInterrupt: 